In [ ]:
!pip install llama-index-core llama-index llama-index-postprocessor-sbert-rerank cohere llama-index-postprocessor-cohere-rerank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 18.2 MB/s eta 0:00:00


In [ ]:
LLAMA_CLOUD_API_KEY = 'XXX-XXX-XXX'
OPENAI_API_KEY = 'XXX-XXX-XXX'
COHERE_API_KEY = 'XXX-XXX-XXX'

In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()

os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["COHERE_API_KEY"] = OPENAI_API_KEY

In [ ]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(extract_charts=True,
                    language="en",
                    model="gpt-3.5-turbo",
                    invalidate_cache=True)
json_objs = parser.get_json_result("/content/handwritten.pdf")

Started parsing the file under job_id 61357d9f-eba4-48c7-b1c7-9d2da84949d1


In [ ]:
print(json_objs[0]["pages"][0]['text'])

   ProQou  Dekuskouu wrike OA Joy Preara Cocatoon
    Iarauslef   Wlock kov
    20-k_inleruae meuory (ocaliou
    Km: Vo                o1 iwlerne
   Dkuz"             adurssij    mdes
        z21ou / Varioua            Qud
           Vuquuovd asswd
   lloadwwe_aud Sofhue Rquremeuls Zm p
   scpkeoe
    Algnln
     Slerf
    7 Load coun Voluo i fedskef
    3 Pou F Ro 06 itesuol mouud  ZocaZrow 3oh
    1 Rout Ri                    OOcoo~_1l
    5    leo              pould  bey   +0
      Lofd           Accuuuuubi  Qo MQuuod pouded_
   L4e2
       hemeu}
csl


In [ ]:
_ = parser.get_charts(json_objs, download_path="charts")

In [ ]:
_ = parser.get_images(json_objs, download_path="images")

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.2)

In [ ]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(
    model="gpt-3.5-turbo",
    language="en",
    extract_charts=True,
    extract_images=True,
    extract_tables=True,
    result_type="markdown",
)

In [ ]:
documents = await parser.aload_data("/content/handwritten.pdf")

Started parsing the file under job_id ed3cecc3-34f0-4cd0-b208-93081d94ddb7


In [ ]:
import nest_asyncio

nest_asyncio.apply()

from llama_index.core.node_parser import (
    MarkdownElementNodeParser,
    SentenceSplitter,
)

node_parser = MarkdownElementNodeParser(num_workers=8)
nodes = node_parser.get_nodes_from_documents(documents)
nodes, objects = node_parser.get_nodes_and_objects(nodes)

nodes = SentenceSplitter(chunk_size=512, chunk_overlap=20).get_nodes_from_documents(
    nodes
)

0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [ ]:
from llama_index.core import VectorStoreIndex, SummaryIndex

vector_index = VectorStoreIndex(nodes=nodes)
summary_index = SummaryIndex(nodes=nodes)

In [ ]:
from llama_index.agent.openai import OpenAIAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.postprocessor.cohere_rerank import CohereRerank

tools = [
    QueryEngineTool(
        vector_index.as_query_engine(
            similarity_top_k=8,
            node_postprocessors=[CohereRerank(top_n=3)]
        ),
        metadata=ToolMetadata(
            name="search",
            description="Search the document, pass the entire user message in the query",
        ),
    ),
    QueryEngineTool(
        summary_index.as_query_engine(),
        metadata=ToolMetadata(
            name="summarize",
            description="Summarize the document using the user message",
        ),
    ),
]

agent = OpenAIAgent.from_tools(tools=tools, verbose=True)

In [ ]:
resp = agent.chat("What is the summary of the paper?")

Added user message to memory: What is the summary of the paper?
=== Calling Function ===
Calling function: summarize with args: {"input":"What is the summary of the paper?"}
Got output: The paper discusses various technical details related to a system or software, including memory specifications, internal components, load counts, routes, and system requirements. It also mentions issues related to indirect addressing and stopping processes.



In [ ]:
print(str(resp))

The summary of the paper includes technical information, system requirements, memory details, internal interfaces, software requirements, load counts, routes, and addresses.


In [ ]:
resp = agent.chat("How do the authors evaluate their work?")

Added user message to memory: How do the authors evaluate their work?
=== Calling Function ===
Calling function: search with args: {"input":"evaluation"}
Got output: Error: status_code: 401, body: {'message': 'invalid api token'}



In [ ]:
print(str(resp))

I encountered an error while trying to retrieve information on how the authors evaluate their work. Unfortunately, I am unable to provide the specific details at the moment.
